# Initialize the local SpecMine data cache

Set `SPEC_SAMPLE_SIZE` and `SEED` below, then run this notebook to cache the metadata and precomputed content-feature parquet files and download that many random raw spec files into `notebooks/data`. The sample is drawn from the full SpecMine metadata, with duplicate spec IDs removed and at most one randomly selected spec per repository. If a spec download fails, a different repository/spec is tried to fill the requested count. Downloaded files are reused on later runs. Other notebooks use local-only mode for parquet data and do not silently contact Hugging Face.

In [ ]:
import sys
from pathlib import Path

project_root = next(
    (
        path
        for path in (Path.cwd(), *Path.cwd().parents)
        if (path / "src" / "specmine").is_dir()
    ),
    None,
)
if project_root is None:
    raise FileNotFoundError(
        "Could not find src/specmine. Open this notebook from the project "
        "folder or one of its subfolders."
    )
sys.path.insert(0, str(project_root / "src"))

from specmine.data import (
    download_dataset_parquets,
    download_random_spec_sample,
    load_data,
)

DATA_DIR = project_root / "notebooks" / "data"
SPEC_SAMPLE_SIZE = 500
SEED = 45

print(f"SpecMine cache directory: {DATA_DIR}")

In [ ]:
downloaded_files = download_dataset_parquets(DATA_DIR)
for name, path in downloaded_files.items():
    size_mb = path.stat().st_size / (1024 * 1024)
    print(f"{name}: {path} ({size_mb:.1f} MiB)")

metadata = load_data(curated_only=False, local_data_only=True, data_dir=DATA_DIR)
if metadata is None or metadata.empty:
    raise RuntimeError("The local SpecMine metadata table is empty or unavailable.")

sampled_specs = download_random_spec_sample(
    metadata,
    SPEC_SAMPLE_SIZE,
    seed=SEED,
    data_dir=DATA_DIR,
)
if len(sampled_specs) != SPEC_SAMPLE_SIZE:
    raise RuntimeError(
        f"Expected {SPEC_SAMPLE_SIZE} downloaded specs, got {len(sampled_specs)}."
    )
if sampled_specs["file_url_sha16"].nunique() != SPEC_SAMPLE_SIZE:
    raise RuntimeError("The sample contains duplicate spec IDs.")
if sampled_specs["repo_id"].nunique() != SPEC_SAMPLE_SIZE:
    raise RuntimeError("The sample contains duplicate repository IDs.")

manifest_path = DATA_DIR / "spec_sample.csv"
sampled_specs.to_csv(manifest_path, index=False)
print(
    f"Downloaded/reused {len(sampled_specs)} distinct specs from {sampled_specs['repo_id'].nunique()} distinct repositories."
)
print(f"Sample manifest: {manifest_path}")
print("Local SpecMine metadata, features, and sampled Markdown files are ready.")